In [30]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/processed")

print("Working directory:", Path.cwd())
print("Data directory:", DATA_DIR.resolve())

Working directory: c:\Users\HP\OneDrive\Documents\GitHub\warehouse-slotting-optimization\notebooks
Data directory: C:\Users\HP\OneDrive\Documents\GitHub\warehouse-slotting-optimization\data\processed


In [31]:
# Required PRODUCT information before slot optimization.
BASE_COLUMNS = [
    "StockCode",
    "order_frequency",
    "total_quantity",
    "avg_quantity_per_order",
    "related_product_count",
    "cluster_name",
    "recommended_zone",
]

# Preferred files, in order.
candidate_names = [
    "final_slot_recommendations.csv",
    "final_product_features.csv",
    "product_features.csv",
    "product_locations.csv",
]

candidate_files = [
    DATA_DIR / name
    for name in candidate_names
    if (DATA_DIR / name).exists()
]

if not candidate_files:
    raise FileNotFoundError(
        "No suitable product CSV was found in "
        f"{DATA_DIR.resolve()}. Found files: "
        + ", ".join(p.name for p in DATA_DIR.glob("*.csv"))
    )

selected_file = None
selected_df = None

for file in candidate_files:
    temp = pd.read_csv(file)
    missing_base = [c for c in BASE_COLUMNS if c not in temp.columns]

    if not missing_base:
        selected_file = file
        selected_df = temp
        break

if selected_file is None:
    details = []
    for file in candidate_files:
        temp = pd.read_csv(file, nrows=5)
        missing = [c for c in BASE_COLUMNS if c not in temp.columns]
        details.append(f"{file.name}: missing {missing}")

    raise KeyError(
        "No product CSV contains the required base columns.\n"
        + "\n".join(details)
    )

products = selected_df.copy()

print("Selected product file:", selected_file)
print("Products shape:", products.shape)
print("Columns:")
print(products.columns.tolist())


Selected product file: ..\data\processed\final_slot_recommendations.csv
Products shape: (3922, 14)
Columns:
['StockCode', 'order_frequency', 'total_quantity', 'avg_quantity_per_order', 'related_product_count', 'cluster_name', 'recommended_zone', 'optimized_slot', 'optimized_x', 'optimized_y', 'picking_priority', 'current_distance', 'optimized_distance', 'actual_distance_saved']


In [32]:
# Load warehouse layout.
warehouse_file = DATA_DIR / "warehouse_slots.csv"

if not warehouse_file.exists():
    raise FileNotFoundError(
        f"Missing warehouse file: {warehouse_file.resolve()}"
    )

warehouse_slots = pd.read_csv(warehouse_file)

required_warehouse = ["slot", "x", "y"]
missing_warehouse = [
    c for c in required_warehouse
    if c not in warehouse_slots.columns
]

if missing_warehouse:
    raise KeyError(
        "warehouse_slots.csv is missing: "
        + ", ".join(missing_warehouse)
    )

print("Warehouse shape:", warehouse_slots.shape)
print(warehouse_slots.head())


Warehouse shape: (25, 3)
  slot  x  y
0   A1  0  0
1   A2  0  1
2   A3  0  2
3   A4  0  3
4   A5  0  4


In [33]:
# Clean identifiers.
products["StockCode"] = (
    products["StockCode"]
    .astype(str)
    .str.strip()
)

warehouse_slots["slot"] = (
    warehouse_slots["slot"]
    .astype(str)
    .str.strip()
)

# Remove ONLY old optimization outputs.
# These will be rebuilt from scratch.
old_optimization_columns = [
    "optimized_slot",
    "optimized_x",
    "optimized_y",
    "optimized_distance",
    "actual_distance_saved",
    "slot",
    "zone",
    "slot_distance",
]

products = products.drop(
    columns=[
        c for c in old_optimization_columns
        if c in products.columns
    ],
    errors="ignore"
)

# Numeric base fields.
numeric_base = [
    "order_frequency",
    "total_quantity",
    "avg_quantity_per_order",
    "related_product_count",
]

for col in numeric_base:
    products[col] = pd.to_numeric(
        products[col],
        errors="coerce"
    ).fillna(0)

# Recommended zone.
products["recommended_zone"] = (
    products["recommended_zone"]
    .fillna("Zone A")
    .astype(str)
    .str.strip()
)

# Preserve existing picking_priority if present.
# Only create a fallback if the source truly does not have it.
if "picking_priority" in products.columns:
    products["picking_priority"] = pd.to_numeric(
        products["picking_priority"],
        errors="coerce"
    )

    # Fill only missing priority values.
    missing_priority = products["picking_priority"].isna()
    products.loc[missing_priority, "picking_priority"] = (
        products.loc[missing_priority, "order_frequency"]
        * products.loc[missing_priority, "total_quantity"]
    )
else:
    products["picking_priority"] = (
        products["order_frequency"]
        * products["total_quantity"]
    )

# Preserve existing current_distance if present.
# Your previous project output used 60 as the baseline.
if "current_distance" in products.columns:
    products["current_distance"] = pd.to_numeric(
        products["current_distance"],
        errors="coerce"
    ).fillna(60.0)
else:
    products["current_distance"] = 60.0

print("Missing priority:", products["picking_priority"].isna().sum())
print("Missing current distance:", products["current_distance"].isna().sum())
print("Product count:", len(products))


Missing priority: 0
Missing current distance: 0
Product count: 3922


In [34]:
zone_mapping = {
    "A": "Zone A",
    "B": "Zone B",
    "C": "Zone C",
    "D": "Zone D",
    "E": "Zone E",
}

warehouse_slots["zone"] = (
    warehouse_slots["slot"]
    .str[0]
    .str.upper()
    .map(zone_mapping)
)

if warehouse_slots["zone"].isna().any():
    bad_slots = warehouse_slots.loc[
        warehouse_slots["zone"].isna(),
        "slot"
    ].tolist()
    raise ValueError(
        "Could not determine zone for slots: "
        + str(bad_slots)
    )

warehouse_slots["x"] = pd.to_numeric(
    warehouse_slots["x"],
    errors="coerce"
)

warehouse_slots["y"] = pd.to_numeric(
    warehouse_slots["y"],
    errors="coerce"
)

if warehouse_slots[["x", "y"]].isna().any().any():
    raise ValueError("Warehouse x/y contains missing or invalid values.")

warehouse_slots["slot_distance"] = (
    warehouse_slots["x"].abs()
    + warehouse_slots["y"].abs()
)

warehouse_slots = warehouse_slots.drop_duplicates(
    subset=["slot"]
).sort_values(
    ["zone", "slot_distance", "x", "y", "slot"]
).reset_index(drop=True)

print(
    warehouse_slots[
        ["slot", "x", "y", "zone", "slot_distance"]
    ].to_string(index=False)
)


slot  x  y   zone  slot_distance
  A1  0  0 Zone A              0
  A2  0  1 Zone A              1
  A3  0  2 Zone A              2
  A4  0  3 Zone A              3
  A5  0  4 Zone A              4
  B1  1  0 Zone B              1
  B2  1  1 Zone B              2
  B3  1  2 Zone B              3
  B4  1  3 Zone B              4
  B5  1  4 Zone B              5
  C1  2  0 Zone C              2
  C2  2  1 Zone C              3
  C3  2  2 Zone C              4
  C4  2  3 Zone C              5
  C5  2  4 Zone C              6
  D1  3  0 Zone D              3
  D2  3  1 Zone D              4
  D3  3  2 Zone D              5
  D4  3  3 Zone D              6
  D5  3  4 Zone D              7
  E1  4  0 Zone E              4
  E2  4  1 Zone E              5
  E3  4  2 Zone E              6
  E4  4  3 Zone E              7
  E5  4  4 Zone E              8


In [35]:
products = products.sort_values(
    by="picking_priority",
    ascending=False,
    kind="stable"
).reset_index(drop=True)

print(
    products[
        [
            "StockCode",
            "recommended_zone",
            "picking_priority",
            "current_distance"
        ]
    ].head(20).to_string(index=False)
)


StockCode recommended_zone  picking_priority  current_distance
   85099B           Zone A         101047019                60
   85123A           Zone A          82734918                60
    22197           Zone A          79202016                60
    84879           Zone A          52906710                60
    21212           Zone A          48042720                60
    47566           Zone A          30806855                60
    23084           Zone A          30554566                60
    20725           Zone A          30411080                60
    84077           Zone A          29398785                60
    22423           Zone A          27535788                60
    22386           Zone A          26123664                60
    23203           Zone A          25217035                60
    22178           Zone A          25173175                60
    22086           Zone A          22421640                60
    22469           Zone A          21573563           

In [36]:
def assign_slots_by_zone(products_df, warehouse_df):
    result = products_df.copy()

    result["optimized_slot"] = pd.NA

    zones = sorted(
        result["recommended_zone"]
        .dropna()
        .unique()
    )

    for zone in zones:

        product_indices = result.index[
            result["recommended_zone"] == zone
        ].tolist()

        zone_slots = warehouse_df[
            warehouse_df["zone"] == zone
        ].sort_values(
            ["slot_distance", "x", "y", "slot"]
        )

        available_slots = zone_slots["slot"].tolist()

        if not available_slots:
            raise ValueError(
                f"No warehouse slots available for {zone}."
            )

        for position, product_index in enumerate(
            product_indices
        ):
            slot_index = position % len(available_slots)

            result.loc[
                product_index,
                "optimized_slot"
            ] = available_slots[slot_index]

    return result


optimized = assign_slots_by_zone(
    products,
    warehouse_slots
)

missing_slots = optimized["optimized_slot"].isna().sum()

print("Total products:", len(optimized))
print("Missing optimized slots:", missing_slots)

if missing_slots != 0:
    raise ValueError(
        f"{missing_slots} products have no optimized slot."
    )


Total products: 3922
Missing optimized slots: 0


In [37]:
print(
    optimized[
        [
            "StockCode",
            "recommended_zone",
            "picking_priority",
            "optimized_slot"
        ]
    ].head(30).to_string(index=False)
)


StockCode recommended_zone  picking_priority optimized_slot
   85099B           Zone A         101047019             A1
   85123A           Zone A          82734918             A2
    22197           Zone A          79202016             A3
    84879           Zone A          52906710             A4
    21212           Zone A          48042720             A5
    47566           Zone A          30806855             A1
    23084           Zone A          30554566             A2
    20725           Zone A          30411080             A3
    84077           Zone A          29398785             A4
    22423           Zone A          27535788             A5
    22386           Zone A          26123664             A1
    23203           Zone A          25217035             A2
    22178           Zone A          25173175             A3
    22086           Zone A          22421640             A4
    22469           Zone A          21573563             A5
    21977           Zone A          2156

In [38]:
slot_lookup = (
    warehouse_slots[
        [
            "slot",
            "x",
            "y",
            "slot_distance"
        ]
    ]
    .drop_duplicates("slot")
    .rename(
        columns={
            "x": "optimized_x",
            "y": "optimized_y",
            "slot_distance": "optimized_distance",
        }
    )
)

optimized = optimized.merge(
    slot_lookup,
    left_on="optimized_slot",
    right_on="slot",
    how="left",
    validate="many_to_one"
)

optimized = optimized.drop(
    columns=["slot"],
    errors="ignore"
)

optimized["optimized_distance"] = pd.to_numeric(
    optimized["optimized_distance"],
    errors="coerce"
)

missing_distance = optimized[
    "optimized_distance"
].isna().sum()

print("Missing optimized distances:", missing_distance)

if missing_distance != 0:
    raise ValueError(
        "Some optimized slots could not be mapped to a distance."
    )


Missing optimized distances: 0


In [39]:
optimized["current_distance"] = pd.to_numeric(
    optimized["current_distance"],
    errors="coerce"
).fillna(60.0)

optimized["actual_distance_saved"] = (
    optimized["current_distance"]
    - optimized["optimized_distance"]
)

print(
    optimized[
        [
            "StockCode",
            "current_distance",
            "optimized_slot",
            "optimized_distance",
            "actual_distance_saved"
        ]
    ].head(20).to_string(index=False)
)


StockCode  current_distance optimized_slot  optimized_distance  actual_distance_saved
   85099B                60             A1                   0                     60
   85123A                60             A2                   1                     59
    22197                60             A3                   2                     58
    84879                60             A4                   3                     57
    21212                60             A5                   4                     56
    47566                60             A1                   0                     60
    23084                60             A2                   1                     59
    20725                60             A3                   2                     58
    84077                60             A4                   3                     57
    22423                60             A5                   4                     56
    22386                60             A1            

In [40]:
final_columns = [
    "StockCode",
    "order_frequency",
    "total_quantity",
    "avg_quantity_per_order",
    "related_product_count",
    "cluster_name",
    "recommended_zone",
    "optimized_slot",
    "optimized_x",
    "optimized_y",
    "picking_priority",
    "current_distance",
    "optimized_distance",
    "actual_distance_saved",
]

missing_final = [
    c for c in final_columns
    if c not in optimized.columns
]

if missing_final:
    raise KeyError(
        "Missing final columns: "
        + ", ".join(missing_final)
    )

final_slot_recommendations = optimized[
    final_columns
].copy()

# Keep one row per StockCode.
duplicate_codes = (
    final_slot_recommendations["StockCode"]
    .duplicated()
    .sum()
)

if duplicate_codes:
    raise ValueError(
        f"Found {duplicate_codes} duplicate StockCode rows."
    )

OUTPUT_FILE = DATA_DIR / "final_slot_recommendations.csv"

final_slot_recommendations.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved:", OUTPUT_FILE.resolve())
print("Rows:", len(final_slot_recommendations))
print(
    "Unique StockCodes:",
    final_slot_recommendations["StockCode"].nunique()
)


Saved: C:\Users\HP\OneDrive\Documents\GitHub\warehouse-slotting-optimization\data\processed\final_slot_recommendations.csv
Rows: 3922
Unique StockCodes: 3922


In [42]:
checks = {
    "Missing StockCode":
        final_slot_recommendations["StockCode"].isna().sum(),

    "Missing recommended_zone":
        final_slot_recommendations["recommended_zone"].isna().sum(),

    "Missing optimized_slot":
        final_slot_recommendations["optimized_slot"].isna().sum(),

    "Missing optimized_distance":
        final_slot_recommendations["optimized_distance"].isna().sum(),

    "Missing picking_priority":
        final_slot_recommendations["picking_priority"].isna().sum(),

    "Missing current_distance":
        final_slot_recommendations["current_distance"].isna().sum(),

    "Duplicate StockCode":
        final_slot_recommendations["StockCode"].duplicated().sum(),
}

for name, value in checks.items():
    print(f"{name}: {value}")

if any(value != 0 for value in checks.values()):
    raise ValueError(
        "Validation FAILED. Fix the above issue before using Streamlit."
    )

print("\n===================================")
print("ALL VALIDATION CHECKS PASSED")
print("===================================")


Missing StockCode: 0
Missing recommended_zone: 0
Missing optimized_slot: 0
Missing optimized_distance: 0
Missing picking_priority: 0
Missing current_distance: 0
Duplicate StockCode: 0

ALL VALIDATION CHECKS PASSED


In [43]:
test_codes = [
    "85123A",
    "85099B",
    "22423",
    "47566",
    "20725",
]

test_result = final_slot_recommendations[
    final_slot_recommendations["StockCode"].isin(test_codes)
][
    [
        "StockCode",
        "cluster_name",
        "recommended_zone",
        "picking_priority",
        "optimized_slot",
        "current_distance",
        "optimized_distance",
        "actual_distance_saved",
    ]
]

print(test_result.to_string(index=False))


StockCode         cluster_name recommended_zone  picking_priority optimized_slot  current_distance  optimized_distance  actual_distance_saved
   85099B High-Volume Products           Zone A         101047019             A1                60                   0                     60
   85123A High-Volume Products           Zone A          82734918             A2                60                   1                     59
    47566 High-Volume Products           Zone A          30806855             A1                60                   0                     60
    20725 High-Volume Products           Zone A          30411080             A3                60                   2                     58
    22423 High-Volume Products           Zone A          27535788             A5                60                   4                     56


In [44]:
saved_check = pd.read_csv(OUTPUT_FILE)

print("Reloaded CSV shape:", saved_check.shape)
print("Reloaded columns:", saved_check.columns.tolist())

print(
    saved_check[
        saved_check["StockCode"].isin(
            ["85123A", "85099B", "22423", "47566", "20725"]
        )
    ][
        [
            "StockCode",
            "recommended_zone",
            "optimized_slot",
            "optimized_distance",
            "actual_distance_saved"
        ]
    ].to_string(index=False)
)

print("\nFINAL CSV IS READY FOR STREAMLIT.")


Reloaded CSV shape: (3922, 14)
Reloaded columns: ['StockCode', 'order_frequency', 'total_quantity', 'avg_quantity_per_order', 'related_product_count', 'cluster_name', 'recommended_zone', 'optimized_slot', 'optimized_x', 'optimized_y', 'picking_priority', 'current_distance', 'optimized_distance', 'actual_distance_saved']
StockCode recommended_zone optimized_slot  optimized_distance  actual_distance_saved
   85099B           Zone A             A1                   0                     60
   85123A           Zone A             A2                   1                     59
    47566           Zone A             A1                   0                     60
    20725           Zone A             A3                   2                     58
    22423           Zone A             A5                   4                     56

FINAL CSV IS READY FOR STREAMLIT.
